# STAGE 1: Kaggle T4 Training - 30K Hard Subset

**Target**: Warm-start mAP 80% → 82-86%

**Strategy**: Freeze Swin vision encoder → Full-FT BERT cross-attn + heads (box/anomaly/pose) + XBM queue

---
## 1. Setup, Data Scan, Extraction & Config (ALL IN ONE CELL)

In [ ]:
import os, sys, shutil, json, yaml, subprocess, tarfile
from pathlib import Path

# ── Install zstandard for .tar.zst extraction ──
try:
    import zstandard as zstd
    print('zstandard OK')
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'zstandard'], check=True, timeout=120)
    import zstandard as zstd
    print('zstandard installed')

# ── Clone repo if not exists ──
CODE = Path('/kaggle/working/Model_XVLM_Training')
if not CODE.exists():
    print('Cloning repo...')
    subprocess.run(['git', 'clone', 'https://github.com/Khanhhh239/Model_XVLM_Training.git', str(CODE)],
                   check=True, capture_output=True, text=True, timeout=300)
    print('Repo cloned')
print('Installing deps...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'albumentations', 'pyarrow'], check=True, timeout=180)
os.chdir(str(CODE))
print(f'CWD: {os.getcwd()}')

# ── Scan ALL files across ALL Kaggle datasets ──
input_dir = Path('/kaggle/input')
all_files = []
for d in sorted(input_dir.iterdir()):
    for f in d.rglob('*'):
        if f.is_file():
            all_files.append(f)
print(f'Found {len(all_files)} files in {len(list(input_dir.iterdir()))} dataset(s)')

# ── Categorize ──
tar_zst = [f for f in all_files if '.tar' in f.name and '.zst' in f.name]
pths = [f for f in all_files if f.suffix == '.pth']
boxes_list = [f for f in all_files if 'boxes' in f.name.lower() and f.suffix == '.jsonl']

print(f'  Archives: {len(tar_zst)}')
print(f'  Checkpoints: {len(pths)}')
print(f'  Boxes files: {len(boxes_list)}')

# ── Create data dirs ──
Path('data/checkpoints').mkdir(parents=True, exist_ok=True)

# ── Copy checkpoint ──
if pths:
    src = pths[0]
    shutil.copy(str(src), 'data/checkpoints/best.pth')
    print(f'Checkpoint: {src.name} ({src.stat().st_size/1024**2:.1f}MB)')
else:
    print('WARNING: No .pth checkpoint found!')

# ── Copy boxes ──
if boxes_list:
    shutil.copy(str(boxes_list[0]), f'data/{boxes_list[0].name}')
    print(f'Boxes: {boxes_list[0].name}')

# ── Extract archive ──
def extract_zst_tar(archive_path, dst='data', max_strip=3):
    for strip in range(max_strip + 1):
        cnt = 0
        dctx = zstd.ZstdDecompressor()
        with open(str(archive_path), 'rb') as f:
            with dctx.stream_reader(f) as reader:
                with tarfile.open(fileobj=reader, mode='r|') as tar:
                    for m in tar:
                        if not m.isfile():
                            continue
                        parts = m.name.split('/', strip)
                        m.name = '/'.join(parts[strip:]) if len(parts) > strip else parts[-1]
                        tar.extract(m, path=dst)
                        cnt += 1
        if cnt > 0:
            print(f'  strip={strip}: {cnt} files')
            return cnt
        print(f'  strip={strip}: 0 files (trying next...)')
    return 0

need_extract = not any('train_30k_hard' in p.name for p in Path('data').rglob('*.jsonl'))
if tar_zst and need_extract:
    a = tar_zst[0]
    print(f'Extracting {a.name} ({a.stat().st_size/1e9:.1f}GB)...')
    n = extract_zst_tar(a)
    if n == 0:
        raise RuntimeError(f'Extraction produced 0 files from {a.name}')
elif not tar_zst:
    print('No .tar.zst archive found in any dataset')
else:
    print('Extraction skipped (data already present)')

# ── Print extracted structure ──
print('\nContents of data/:')
for root, dirs, fnames in os.walk('data'):
    depth = root.replace('data', '').count(os.sep)
    if depth > 2:
        continue
    prefix = '  ' * depth
    print(f'{prefix}{os.path.basename(root) or "data"}/' if depth > 0 else f'{os.path.basename(root) or "data"}/')
    for fn in sorted(fnames)[:10]:
        fp = Path(root, fn)
        label = fn if fp.stat().st_size < 1e6 else f'{fn} ({fp.stat().st_size/1024**2:.1f}MB)'
        print(f'{prefix}  {label}')
    if len(fnames) > 10:
        print(f'{prefix}  ... ({len(fnames)} files)')

# ── Discover all paths ──
manifest = None
for p in Path('data').rglob('*.jsonl'):
    if 'boxes' not in p.name and 'train' in p.name:
        manifest = str(p)
        break
if not manifest:
    for p in Path('data').rglob('*.jsonl'):
        if 'boxes' not in p.name:
            manifest = str(p)
            break

webp_dirs = sorted(set(p.parent for p in Path('data').rglob('*.webp')))
img_root = str(webp_dirs[0]) if webp_dirs else None

vitpose = None
for p in Path('data').rglob('*.json'):
    if 'vitpose' in p.name.lower():
        vitpose = str(p)
        break

boxes = None
for p in Path('data').rglob('*.jsonl'):
    if 'boxes' in p.name.lower():
        boxes = str(p)
        break

ckpt_path = 'data/checkpoints/best.pth'
ckpt_ok = Path(ckpt_path).exists()

print('\n=== DISCOVERED PATHS ===')
print(f'  manifest: {manifest or "NOT FOUND"}')
print(f'  img_root: {img_root or "NOT FOUND"}')
if img_root:
    print(f'    -> {len(list(Path(img_root).rglob("*.webp")))} webp files')
print(f'  vitpose:  {vitpose or "NOT FOUND"}')
print(f'  boxes:    {boxes or "NOT FOUND"}')
print(f'  ckpt:     {ckpt_path} ({"OK" if ckpt_ok else "NOT FOUND"})')

# ── Validate ──
missing = []
if not manifest: missing.append('manifest (jsonl)')
if not img_root: missing.append('img_root (webp dir)')
if not ckpt_ok: missing.append('checkpoint (best.pth)')
if missing:
    raise FileNotFoundError(f'FATAL - Missing: {missing}. Check your Kaggle datasets!')
print('All required paths OK!')

# ── Create runtime config ──
with open('configs/stage1_30k_kaggle_t4.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['data']['manifest'] = manifest
cfg['data']['image_root'] = str(Path(img_root)) + '/'
cfg['data']['vitpose_json'] = vitpose or ''
cfg['data']['boxes_json'] = boxes or ''
cfg['model']['checkpoint'] = ckpt_path
if not vitpose:
    cfg['model']['pose_enabled'] = False
    print('Pose: disabled (no vitpose file)')
if not boxes:
    cfg['model']['bbox_enabled'] = False
    print('BBox: disabled (no boxes file)')
cfg['train']['batch_size'] = 16
cfg['train']['num_workers'] = 2
cfg['train']['amp_dtype'] = 'fp16'
cfg['train']['grad_checkpointing'] = True

cfg_path = 'configs/stage1_kaggle_runtime.yaml'
with open(cfg_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)
print(f'\nConfig saved: {cfg_path}')
print('  manifest: ' + cfg['data']['manifest'])
print('  image_root: ' + cfg['data']['image_root'])
print('  vitpose_json: ' + cfg['data']['vitpose_json'])
print('  boxes_json: ' + cfg['data']['boxes_json'])
print('  checkpoint: ' + cfg['model']['checkpoint'])

---
## 2. Verify All Paths

In [ ]:
with open('configs/stage1_kaggle_runtime.yaml') as f:
    cfg = yaml.safe_load(f)

checks = [
    ('Manifest', cfg['data']['manifest']),
    ('Image root', cfg['data']['image_root']),
    ('Checkpoint', cfg['model']['checkpoint']),
]
if cfg['data'].get('vitpose_json'): checks.append(('VitPose', cfg['data']['vitpose_json']))
if cfg['data'].get('boxes_json'): checks.append(('Boxes', cfg['data']['boxes_json']))

all_ok = True
for name, path in checks:
    p = Path(path)
    if not path or not p.exists():
        print(f'  MISSING  {name}: {path}')
        all_ok = False
    else:
        sz = f'{p.stat().st_size/1024**2:.1f}MB' if p.is_file() else f'{len(list(p.rglob("*")))} files'
        print(f'  OK  {name}: {path} ({sz})')

if not all_ok:
    raise FileNotFoundError('Fix missing paths before training!')
print('\nAll paths OK! Ready to train.')

---
## 2b. Build Training Manifest (REQUIRED)

Raw `train_30k_hard.jsonl` thiếu `pair_image_id`, `split`, `keypoints` — phải build parquet trước khi train.

In [ ]:
import subprocess

with open('configs/stage1_kaggle_runtime.yaml') as f:
    cfg = yaml.safe_load(f)

# Find raw jsonl (not boxes file)
raw_jsonl = None
for p in Path('data').rglob('*.jsonl'):
    if 'boxes' not in p.name.lower() and 'train' in p.name.lower():
        raw_jsonl = str(p)
        break
if not raw_jsonl:
    for p in Path('data').rglob('*.jsonl'):
        if 'boxes' not in p.name.lower():
            raw_jsonl = str(p)
            break
assert raw_jsonl, 'Cannot find train_30k_hard.jsonl'

webp_root = Path(cfg['data']['image_root'])
manifest_out = 'data/manifest_30k_hard.parquet'

cmd = [
    sys.executable, 'scripts/build_stage1_manifest.py',
    '--jsonl', raw_jsonl,
    '--image-root', str(webp_root),
    '--out', manifest_out,
]
if cfg['data'].get('vitpose_json'):
    cmd.extend(['--vitpose', cfg['data']['vitpose_json']])

print('Building manifest...')
subprocess.run(cmd, check=True)
cfg['data']['manifest'] = manifest_out

with open('configs/stage1_kaggle_runtime.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

import pandas as pd
dfm = pd.read_parquet(manifest_out)
pairs = int(dfm[dfm.split == 'train'].pair_image_id.notna().sum())
kpt_cov = dfm.keypoints.notna().mean() if 'keypoints' in dfm.columns else 0
print(f'Manifest OK: {len(dfm)} rows, pairs(train)={pairs}, keypoints={kpt_cov:.1%}')
print(f'  manifest -> {manifest_out}')
if pairs == 0:
    raise RuntimeError('No pair_image_id pairs — PairBatchSampler will fail!')

---
## 3. Sanity Check (overfit one batch)

In [ ]:
print('Sanity check: overfit one batch for 300 steps...')
!python scripts/train.py --config configs/stage1_kaggle_runtime.yaml --init-from data/checkpoints/best.pth --overfit-one-batch
print('Sanity check passed!')

---
## 4. FULL TRAINING

In [ ]:
resume_flag = ''
if Path('outputs/stage1_30k_t4/last.pth').exists():
    resume_flag = '--resume outputs/stage1_30k_t4/last.pth'
    print('Resuming from last.pth!')
else:
    print('Starting fresh training...')

!python scripts/train.py --config configs/stage1_kaggle_runtime.yaml --init-from data/checkpoints/best.pth --max-hours 11.5 {resume_flag}
print('Training completed!')

---
## 5. Evaluate & Save

In [ ]:
import torch

WORK = Path('/kaggle/working')
best_ckpt = Path('outputs/stage1_30k_t4/best.pth')
last_ckpt = Path('outputs/stage1_30k_t4/last.pth')

if best_ckpt.exists():
    ckpt = torch.load(best_ckpt, map_location='cpu', weights_only=False)
    report = ckpt.get('report', {})
    print('FINAL RESULTS:')
    print(f'  mAP:  {report.get("mAP", 0)*100:.2f}%')
    print(f'  R@1:  {report.get("R@1", 0)*100:.2f}%')
    print(f'  R@5:  {report.get("R@5", 0)*100:.2f}%')
    print(f'  R@10: {report.get("R@10", 0)*100:.2f}%')
    print(f'  MRR:  {report.get("MRR", 0)*100:.2f}%')
    shutil.copy(best_ckpt, WORK / 'stage1_best.pth')
    print(f'Saved to {WORK / "stage1_best.pth"}')
elif last_ckpt.exists():
    print('No best.pth - last.pth exists (can resume)')
else:
    print('No checkpoint found.')

---
## 6. Training Logs

In [ ]:
log_file = Path('outputs/stage1_30k_t4/train.log')
if log_file.exists():
    print(f'Last 80 lines of {log_file}:')
    lines = log_file.read_text().splitlines()
    print('\n'.join(lines[-80:]))
else:
    print('No log file found.')

---
## 7. Save For Next Commit

In [ ]:
import tarfile as tfl
out_dir = Path('outputs/stage1_30k_t4')
if out_dir.exists():
    out = WORK / 'stage1_output.tar.gz'
    with tfl.open(out, 'w:gz') as tar:
        tar.add(out_dir, arcname='stage1_30k_t4')
    print(f'Outputs: {out} ({out.stat().st_size/1024**2:.1f}MB)')
else:
    print('No output directory found.')